* charge sensitivity over charge offset
* variation of Ej/Ec ratio
* verification of analytic results

# Transmon Charge Sensitivity

Here we want to analyze the transmon circuit and its sensitivity towards an external charge offset. This time we want to work in Charge representation. 

In [1]:
from des_scq import models
from des_scq.circuit import Charge,hamiltonianEnergy
from des_scq.optimization import Optimization
from des_scq.discovery import lossTransition
from torch import tensor, float32 as float
from numpy import arange,linspace,std as deviation,mean,sqrt
from des_scq.utils import plotCompare
import pickle
from torch import set_num_threads
set_num_threads(32)

In [ ]:
# set energies for junction and capacitor
Ej = 30.02 ; Ec = 10.2

First build circuit by defining basis size, setting charge representation, and using transmon() function found in models.py.

In [ ]:
# define basis size, representation
basis = [512]
circuit = models.transmon(Charge,basis,Ej,Ec)
print(circuit.circuitComponents())

{'J': 30.020000000000007, 'C': 0.04901960784313726}


Now define charge_offset:

In [1]:
charge_offset = linspace(-2, 2, 10)

NameError: name 'linspace' is not defined

Compute circuit hamiltonian, and with that energies of ground state and first excitation with the function circuitEnergy() found in circuit.py.

In [ ]:
# compute hamiltonians
H_LC = circuit.hamiltonianLC() 
H_J = circuit.hamiltonianJosephson
# compute energies of ground state and first excitation
ground,Ist = hamiltonianEnergy(H_LC+H_J(dict()))[:2]
# compute E (1->0)
Delta = Ist-ground
print('The transition energy from first to ground E(1->0) is', Delta)

Now we want to include the charge offset into the hamiltonian and compute ground, first, second, and third energy level with the charge offset, as well as the enegies $E_{10},E_{20},E_{30}$. We loop over different charge offsets and perform these computation for each offset.

In [ ]:
# empty lists for ground, first, second, third energy level
G,I,II,III = [],[],[],[]
del1, del2, del3 = [],[],[]

for q in charge_offset: # loop over each charge offset
    H_LC = circuit.hamiltonianLC()
    offset = dict([(1,tensor(q))])
    # compute charge offset hamiltonian
    H_off = circuit.hamiltonianChargeOffset(offset)
    H_J = circuit.hamiltonianJosephson
    ground,Ist,IInd,IIIrd = hamiltonianEnergy(H_LC+H_off+H_J(dict()))[:4]
    # compute transition energies
    delta1 = Ist-ground
    delta2 = IInd-ground
    delta3 = IIIrd-ground

    G.append(ground)
    I.append(Ist)
    II.append(IInd)
    III.append(IIIrd)

    del1.append(delta1)
    del2.append(delta2)
    del3.append(delta3)

Plot ground, first, second, and third energy level for all charge offsets.

In [7]:
plots = {'0':G,'I':I,'II':II}

In [ ]:
plotCompare(charge_offset,plots,'Transmon-Offset Charge','n_g','E_m(GHz)',export='pdf',size=(600,800))

Plot energy transitions E10, E20, E30 for all charge offsets.

In [ ]:
plots = {'E10':del1,'E20':del2,'E30':del3}
plotCompare(charge_offset,plots,'Transmon-Offset Charge','$n_g$','$E_m(GHz)$',export='pdf',size=(600,600))

## Ej/Ec variation
We want to analyze the spectrum for different ratios of Josephson and Capacitor energies.

In [ ]:
# set josephson energy and different Ej/Ec ratios
Ej = 30.02
ratio = linspace(1,50,100,endpoint=True)

In [11]:
std = {'std0':[],'std1':[],'std2':[]}
avg = {'mean0':[],'mean1':[],'mean2':[]}
deltaHalf = []

For each ratio Ej/Ec we again calculate ground, first, second, and third energy level with the charge offset, and the enegies $E_{10},E_{20},E_{30}$.

In [ ]:
for r in ratio:
    Ec = Ej/r
    circuit = models.transmon(Charge,basis,Ej,Ec)
    G,I,II = [],[],[]
    for q in charge_offset:
        H_LC = circuit.hamiltonianLC()
        offset = dict([(1,tensor(q))])
        H_off = circuit.hamiltonianChargeOffset(offset)
        H_J = circuit.hamiltonianJosephson
        ground,Ist,IInd = hamiltonianEnergy(H_LC+H_off+H_J(dict()))[:3]
        delta = Ist-ground
        G.append((ground/delta).detach().numpy())
        I.append((Ist/delta).detach().numpy())
        II.append((IInd/delta).detach().numpy())
        if q == .5:
            deltaHalf.append((Ist/delta).detach().numpy())
    # analysis of dispersion profile
    std['std0'].append(deviation(G)) ; avg['mean0'].append(mean(G))
    std['std1'].append(deviation(I)) ; avg['mean1'].append(mean(I))
    std['std2'].append(deviation(II)) ; avg['mean2'].append(mean(II))

In [ ]:
#plots = {'delta Half' : deltaHalf}
plots.update(std)
plots.update(avg)

In [ ]:
plotCompare(ratio,std,'Transmon Insensitivity-Deviation','Ej/Ec',export='pdf',size=(600,800))

In [ ]:
plotCompare(ratio,avg,'Transmon Insensitivity-Mean','Ej/Ec',export='pdf',size=(600,800))